# **Import libraries**

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
import os
from google.generativeai import configure
from sentence_transformers import SentenceTransformer, util
import numpy as np
import pandas as pd
import gradio as gr
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import PromptTemplate

# **Setting API key and PDF path**

In [ ]:
os.environ["GOOGLE_API_KEY"] = "API_KEY"
PDF_PATH = "/content/Sample Employee Handbook.pdf"

# **Load PDF & chunking**

In [ ]:
loader = PyPDFLoader(PDF_PATH)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = loader.load_and_split(text_splitter)

# **Create embeddings and vector store**

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vectorstore = FAISS.from_documents(chunks, embeddings)

# **Setup LLM with a prompt**

In [ ]:
#setting llm model
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)

#prompt creation
prompt = ChatPromptTemplate.from_template("""
 Answer ONLY with the specific information requested from the company policy.
Do not include any explanations, justifications.

Context: {context}

Question: {input}

Answer (only the requested information):
""")

# **Creating QA chain**

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
qa_chain = create_retrieval_chain(retriever, prompt | llm)

# **User question & Response**

In [ ]:
question = input("Enter your company policy question: ")
response = qa_chain.invoke({"input":question})
print("Response:",response["answer"].content)

# **Evaluation on simple QA chain**
Faithfulness, Answer relavancy, Precision, Recall

In [ ]:
# Initialize sentence transformer for similarity calculations
similarity_model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:


# Test data
test_questions = [
    "What is the vacation policy for employees in their first year?",
    "What is the unpaid parental leave policy for school-related events?"
]

ground_truths = [
    "Full-time employees earn 10 days of vacation during their first year after the initial 90-day period.",
    "A parent is entitled to a total of 24 hours of unpaid leave during any 12-month period to attend school-related events for their child. Employees may use accrued vacation or personal leave for this purpose."
]

# Evaluation result storage
faithfulness_scores = []
relevancy_scores = []
context_precision_scores = []
context_recall_scores = []

# Evaluation loop
for i, question in enumerate(test_questions):
    # Get RAG response
    response = qa_chain.invoke({"input": question})
    answer = response["answer"].content
    retrieved_contexts = [doc.page_content for doc in response["context"]]

    # Faithfulness: Answer vs Retrieved Contexts
    answer_embed = similarity_model.encode(answer)
    context_embed = similarity_model.encode(" ".join(retrieved_contexts))
    faithfulness = util.pytorch_cos_sim(answer_embed, context_embed).item()
    faithfulness_scores.append(faithfulness)

    # Relevancy: Question vs Answer
    q_embed = similarity_model.encode(question)
    a_embed = similarity_model.encode(answer)
    relevancy = util.pytorch_cos_sim(q_embed, a_embed).item()
    relevancy_scores.append(relevancy)

    # Context Precision and Recall
    context_embeddings = similarity_model.encode(retrieved_contexts)
    similarities = [util.pytorch_cos_sim(q_embed, ctx).item() for ctx in context_embeddings]
    ranked_indices = np.argsort(similarities)[::-1]


    relevant_indices = [0]
    relevant_retrieved = 0
    precision_at_k = []

    for rank, idx in enumerate(ranked_indices, 1):
        if idx in relevant_indices:
            relevant_retrieved += 1
            precision_at_k.append(relevant_retrieved / rank)

    context_precision = np.mean(precision_at_k) if precision_at_k else 0
    context_recall = relevant_retrieved / len(relevant_indices)

    context_precision_scores.append(context_precision)
    context_recall_scores.append(context_recall)

    # Print per-question results
    print(f"\nQuestion {i + 1}: {question}")
    print(f"- Faithfulness: {faithfulness:.2f}")
    print(f"- Relevancy: {relevancy:.2f}")
    print(f"- Context Precision: {context_precision:.2f}")
    print(f"- Context Recall: {context_recall:.2f}")



# **Different QA Chains**

# **RetrievalQA chain**

In [ ]:
from langchain.chains import RetrievalQA

# Create RetrievalQA chain
retrieval_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)
question = input("Enter your company policy question: ")
response = retrieval_qa_chain.invoke({"query": question})
print("Response:", response["result"])

##**Evaluation on RetrievalQA chain**

In [ ]:
# Evaluation result storage
faithfulness_scores = []
relevancy_scores = []
context_precision_scores = []
context_recall_scores = []

# Loop over each test case
for i, question in enumerate(test_questions):
    print(f"\nEvaluating Q{i+1}: {question}")

    # Get RAG output
    response = retrieval_qa_chain.invoke({"query": question})
    answer = response["result"]
    retrieved_contexts = [doc.page_content for doc in response["source_documents"]]

    # Faithfulness
    answer_embed = similarity_model.encode(answer)
    context_embed = similarity_model.encode(" ".join(retrieved_contexts))
    faithfulness = util.pytorch_cos_sim(answer_embed, context_embed).item()
    faithfulness_scores.append(faithfulness)

    # Relevancy
    q_embed = similarity_model.encode(question)
    a_embed = similarity_model.encode(answer)
    relevancy = util.pytorch_cos_sim(q_embed, a_embed).item()
    relevancy_scores.append(relevancy)

    # Context Precision/Recall
    gt_embed = similarity_model.encode(ground_truths[i])
    context_embeddings = similarity_model.encode(retrieved_contexts)

    # Compare each retrieved context with the ground truth
    similarities = [util.pytorch_cos_sim(gt_embed, ctx).item() for ctx in context_embeddings]
    ranked_indices = np.argsort(similarities)[::-1]


    relevant_indices = [0]

    relevant_retrieved = 0
    precision_at_k = []

    for rank, idx in enumerate(ranked_indices, 1):
        if idx in relevant_indices:
            relevant_retrieved += 1
            precision_at_k.append(relevant_retrieved / rank)

    context_precision = np.mean(precision_at_k) if precision_at_k else 0
    context_recall = relevant_retrieved / len(relevant_indices)

    context_precision_scores.append(context_precision)
    context_recall_scores.append(context_recall)


    print(f"Answer: {answer}")
    print(f"Ground Truth: {ground_truths[i]}")
    print(f"Faithfulness: {faithfulness:.2f}")
    print(f"Relevancy: {relevancy:.2f}")
    print(f"Context Precision: {context_precision:.2f}")
    print(f"Context Recall: {context_recall:.2f}")



**Different document chain unsing in RetrievalQA chain**

**Map_Reduce Document chain**

In [ ]:
map_reduce_qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="map_reduce",
    retriever=retriever
)
result = map_reduce_qa.invoke({"query": "working time"})
print(result["result"])

**Map_Rerank Document chain**

In [ ]:
# For selecting the best answer
map_rerank_qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="map_rerank",
    retriever=retriever
)
result = map_rerank_qa.invoke({"query": "What is the vacation policy?"})
print(result["result"])

**Refine Document chain**

In [ ]:
refine_qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="refine",
    retriever=retriever
)
result = refine_qa.invoke({"query": "vaccation policy"})
print(result["result"])

**RetrievalQAWithSourcesChain**

In [ ]:


# Create custom prompt with source information
template = """Use the following pieces of context to answer the question at the end.
Include the "SOURCE" in your answer when relevant.

{context}

Question: {question}
Answer:"""
QA_PROMPT = PromptTemplate(
    template=template, input_variables=["context", "question"]
)

#creating QA chain
qa_with_sources = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(),
    chain_type_kwargs={"prompt": QA_PROMPT},
    return_source_documents=True
)

#user question & response
question = input("Enter your company policy question: ")
result = qa_with_sources.invoke({"query": question})
print("Answer:", result["result"])
print("Source:", result["source_documents"][0].metadata["source"])

**Evaluation on RetrievalQAWithSourcesChain**

In [ ]:
# Evaluation result storage
faithfulness_scores = []
relevancy_scores = []
context_precision_scores = []
context_recall_scores = []

# Loop over each test case
for i, question in enumerate(test_questions):
    print(f"\nEvaluating Q{i+1}: {question}")

    # Get RAG output
    result = qa_with_sources.invoke({"query": question})
    answer = result["result"]
    retrieved_contexts = [doc.page_content for doc in result["source_documents"]]

    # Faithfulness
    answer_embed = similarity_model.encode(answer)
    context_embed = similarity_model.encode(" ".join(retrieved_contexts))
    faithfulness = util.pytorch_cos_sim(answer_embed, context_embed).item()
    faithfulness_scores.append(faithfulness)

    # Relevancy
    q_embed = similarity_model.encode(question)
    a_embed = similarity_model.encode(answer)
    relevancy = util.pytorch_cos_sim(q_embed, a_embed).item()
    relevancy_scores.append(relevancy)

    # Context Precision/Recall
    gt_embed = similarity_model.encode(ground_truths[i])
    context_embeddings = similarity_model.encode(retrieved_contexts)

    # Compare each retrieved context with the ground truth
    similarities = [util.pytorch_cos_sim(gt_embed, ctx).item() for ctx in context_embeddings]
    ranked_indices = np.argsort(similarities)[::-1]


    relevant_indices = [0]

    relevant_retrieved = 0
    precision_at_k = []

    for rank, idx in enumerate(ranked_indices, 1):
        if idx in relevant_indices:
            relevant_retrieved += 1
            precision_at_k.append(relevant_retrieved / rank)

    context_precision = np.mean(precision_at_k) if precision_at_k else 0
    context_recall = relevant_retrieved / len(relevant_indices)

    context_precision_scores.append(context_precision)
    context_recall_scores.append(context_recall)

    print(f"Answer: {answer}")
    print(f"Ground Truth: {ground_truths[i]}")
    print(f"Faithfulness: {faithfulness:.2f}")
    print(f"Relevancy: {relevancy:.2f}")
    print(f"Context Precision: {context_precision:.2f}")
    print(f"Context Recall: {context_recall:.2f}")

## **Conversational QA**

In [ ]:

# creating memory for chat history
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

#creating conversational_qa
conversational_qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    memory=memory
)

# user question & response
result = conversational_qa({"question": "What is the working time?"})
print(result['answer'])

result = conversational_qa({"question": "part time employee?"})  # Remembers context
print(result['answer'])

## **ChatVectorDBChain**

In [ ]:
# import required libraries
from langchain.chains import ChatVectorDBChain

#creating ChatVectorDBChain
chat_vector_chain = ChatVectorDBChain.from_llm(
    llm=llm,
    vectorstore=vectorstore
)

chat_history = []

# user question & response
result = chat_vector_chain({"question": "What is the working hours?", "chat_history": chat_history})
print(result['answer'])
chat_history.append((result['question'], result['answer']))

result = chat_vector_chain({"question": "part time employee?", "chat_history": chat_history})
print(result['answer'])  # Response considering previous conversation

# **Gradio Chat Interface**

In [ ]:

#Setting API key and PDF path
os.environ["GOOGLE_API_KEY"] = "API_KEY"
PDF_PATH = "/content/Sample Employee Handbook.pdf"

#Initialize system
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vectorstore = FAISS.from_documents(chunks, embeddings)
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)
prompt = ChatPromptTemplate.from_template("""
[Strict Instruction] Answer ONLY with the specific information requested from the Company policy.
Do not include any explanations, justifications, or metadata.

Context: {context}

Question: {input}

Answer (only the requested information):
""")

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
qa_chain = create_retrieval_chain(retriever, prompt | llm)


# Chatbot function
def chat_fn(message, history):
    try:
        response = qa_chain.invoke({"input": message})
        if hasattr(response["answer"], "content"):
            return response["answer"].content
        else:
            return str(response["answer"])
    except Exception as e:
        return f" Error: {str(e)}"

# Launch Chatbot
gr.ChatInterface(chat_fn, title="📄 Company Policy Chatbot", description="Ask questions about your Company policy document").launch()